In [1]:
#1
%matplotlib inline 
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import matplotlib.pyplot as plt
import helper_functions as hlp


torch.set_default_dtype(torch.float64)


T = 60e-6           
B = 2e6             
Fs = 5 * B          
N = int(np.round(T * Fs)) 

N_fft = 2**14


t_np = np.linspace(0, T, N, endpoint=False)
t = torch.tensor(t_np, dtype=torch.float64)

t_norm = t / torch.max(t)

b_slope = B/T
psi = 2 * np.pi * (b_slope/2) * t**2
a = 1
s_base = a * torch.complex(torch.cos(psi), torch.sin(psi))

null_freqs = [0.4e6] 
print(f"Targets defined at: {[f/1e6 for f in null_freqs]} MHz")

# Steering Matrix
steering_vectors = []
for f in null_freqs:
    w = 2 * np.pi * f
    vec = torch.complex(torch.cos(w * t), -torch.sin(w * t))
    steering_vectors.append(vec)

steering_matrix = torch.stack(steering_vectors)


z = torch.tensor(hlp.build_z(a,psi,t,null_freqs))
print(torch._shape_as_tensor(z))

Targets defined at: [0.4] MHz
tensor([600,   1])


In [2]:
from version1 import getphi
phi_start =  torch.tensor(getphi(null_freqs), dtype=torch.float64, requires_grad=True)
print("initial_phi_shape:", phi_start.shape)

phi_start = phi_start.squeeze()
print("initial_phi_shape_after_squeeze:", phi_start.shape)




Null at around 0.40 MHz: Depth = -37.34 dB
initial_phi_shape: torch.Size([600, 1])
initial_phi_shape_after_squeeze: torch.Size([600])


In [ ]:
# יצירת וקטור רנדומלי בטווח [15 ,15-]
#phi_start = (30 * torch.rand(600, dtype=torch.float64) - 15).detach().requires_grad_(True)
phi_start = torch.zeros(N, dtype=torch.float64, requires_grad=True)
print("initial_phi_shape:", phi_start.shape)


initial_phi_shape: torch.Size([600])


In [ ]:
import torch
import torch.fft

def calculate_mf_pslr_islr_torch(signal):
    
    # matched_filter = np.conj(signal[::-1])
    matched_filter = torch.conj(torch.flip(signal, dims=[0]))
    
    # mf = np.convolve(signal, matched_filter, mode='full')
    conv_len = signal.shape[0] + matched_filter.shape[0] - 1
    mf = torch.fft.ifft(torch.fft.fft(signal, n=conv_len) * torch.fft.fft(matched_filter, n=conv_len))
    
    # mf_abs = np.abs(mf)
    mf_abs = torch.abs(mf)
    
    # mf_db = 20 * np.log10(mf_abs + 1e-20)
    mf_db = 20 * torch.log10(mf_abs + 1e-20)
    
    # PSLR Calculation
    center_idx = len(mf_db) // 2
    main_lobe_width_seconds = 2 / B
    margin_samples = int((main_lobe_width_seconds / 2) * Fs)
    left_side = mf_db[:center_idx - margin_samples]
    right_side = mf_db[center_idx + margin_samples:]
    
    # sidelobe_region = np.concatenate((left_side, right_side))
    sidelobe_region = torch.cat((left_side, right_side))
    
    # max_sidelobe = np.max(sidelobe_region)
    max_sidelobe = torch.max(sidelobe_region)
    
    # max_mainlobe = np.max(mf_db)
    max_mainlobe = torch.max(mf_db)
    
    # pslr = max_mainlobe - max_sidelobe
    pslr = max_mainlobe - max_sidelobe

    # ISLR calculation
    # n_main = np.arange(len(mf_db[center_idx - margin_samples : center_idx + margin_samples]))
    n_main = torch.arange(len(mf_db[center_idx - margin_samples : center_idx + margin_samples]), dtype=mf_abs.dtype, device=mf_abs.device)
    
    # n_left = np.arange(len(left_side))
    n_left = torch.arange(len(left_side), dtype=mf_abs.dtype, device=mf_abs.device)
    
    # n_right = np.arange(len(right_side))
    n_right = torch.arange(len(right_side), dtype=mf_abs.dtype, device=mf_abs.device)

    
    main_lobe_linear = mf_abs[center_idx - margin_samples : center_idx + margin_samples]
    left_side_linear = mf_abs[:center_idx - margin_samples]
    right_side_linear = mf_abs[center_idx + margin_samples:] 
    main_lobe_energy = torch.trapezoid(main_lobe_linear ** 2, x=n_main)
    
    # sidelobe_energy = np.trapezoid(left_side_linear ** 2, x=n_left) + np.trapezoid(right_side_linear ** 2, x=n_right)
    sidelobe_energy = torch.trapezoid(left_side_linear ** 2, x=n_left) + torch.trapezoid(right_side_linear ** 2, x=n_right)
    
    islr = 10 * torch.log10(sidelobe_energy / main_lobe_energy + 1e-40)
    return mf_db, pslr, islr

In [15]:
# 2
import copy 

phi = torch.tensor(phi_start, dtype=torch.float64, requires_grad=True)

# Check that phi has the correct shape
if phi.shape != (N,):
    raise ValueError(f"Expected phi to have shape ({N},), but got {phi.shape}")

learning_rate =  0.010640845633353869
iterations = 5000 
beta_smooth = 2.6721252174776165e-05
beta_norm = 0 # 1.0914710219012044e-05
beta_div1 = 0
beta_div2 = 0
beta_pslr = 0.001
beta_islr = 0.001 

optimizer = torch.optim.Adam([phi], lr=learning_rate)

print(f"Starting Multi-Null Optimization ({iterations} iterations)...")
print(f"Saving best model based on Minimum Total Loss (Energy + Regularization)")

loss_history = []


best_loss = float('inf') 
best_phi = None
best_epoch = 0

for i in range(iterations):
    
    correction_phasor = torch.complex(torch.cos(phi), torch.sin(phi))
    s_transmit = s_base * correction_phasor
    
   
    spectral_vals = torch.matmul(steering_matrix, s_transmit)
    energies = torch.abs(spectral_vals)**2
    
     
    loss_nulls = torch.sum(energies) 
    
    diff_phi = phi[1:] - phi[:-1]
    reg_term = beta_smooth * torch.sum(diff_phi**2)
    
    
    loss_norm = beta_norm * torch.sum(phi**2)

    if beta_div1 > 0 or beta_div2 > 0:
        integrals1 = [
            torch.trapezoid(z[:, k] * correction_phasor * t_norm, x=t_norm) 
            for k in range(z.shape[1])
        ]

        loss_div1 = beta_div1 * sum([torch.abs(val)**2 for val in integrals1])


        integrals2 = [
            torch.trapezoid(z[:, k] * correction_phasor * (t_norm**2), x=t_norm) 
            for k in range(z.shape[1])
        ]
        loss_div2 = beta_div2 * sum([torch.abs(val)**2 for val in integrals2])
    else:
        loss_div1 = 0
        loss_div2 = 0
    
    mf_db, pslr, islr = calculate_mf_pslr_islr_torch(s_transmit)

    loss_pslr = -1*beta_pslr * pslr
    loss_islr = beta_islr * islr 



    loss = loss_norm  + loss_div1 + loss_div2  + loss_nulls+ loss_pslr + loss_islr + reg_term
    
    
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_phi = phi.clone() 
        best_epoch = i
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    
    
    if i % 1000 == 0:
        print(f"Iter {i}: Total Loss = {loss.item():.6f}")
       
        max_depth = 10 * np.log10(torch.max(energies).item() + 1e-40)
        print(f"   Worst Null Depth: {max_depth:.2f} dB")

print("\n--- Optimization Finished ---")
print(f"Restoring best model from Epoch {best_epoch} (Total Loss: {best_loss:.6f})")


with torch.no_grad():
    phi.copy_(best_phi)



final_spectral = torch.matmul(steering_matrix, s_base * torch.complex(torch.cos(phi), torch.sin(phi)))
final_energies = torch.abs(final_spectral)**2
print("Null Depths in Selected Model:")
for k, f in enumerate(null_freqs):

    e_db = 10 * np.log10(final_energies[k].item())
    print(f"Freq {f/1e6} MHz: {e_db:.2f} dB")

C:\Users\User\AppData\Local\Temp\ipykernel_11800\2934180478.py:4: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).



Starting Multi-Null Optimization (5000 iterations)...
Saving best model based on Minimum Total Loss (Energy + Regularization)
Iter 0: Total Loss = 0.411155
   Worst Null Depth: -3.63 dB
Iter 1000: Total Loss = -0.022609
   Worst Null Depth: -57.32 dB
Iter 2000: Total Loss = -0.022676
   Worst Null Depth: -51.47 dB
Iter 3000: Total Loss = -0.021173
   Worst Null Depth: -28.00 dB
Iter 4000: Total Loss = -0.022308
   Worst Null Depth: -32.81 dB

--- Optimization Finished ---
Restoring best model from Epoch 4958 (Total Loss: -0.022903)
Null Depths in Selected Model:
Freq 0.4 MHz: -70.87 dB


In [ ]:
#3 Showing the Null
null_freqs_np = np.array(null_freqs) 


with torch.no_grad():
    correction_phasor = torch.complex(torch.cos(best_phi), torch.sin(best_phi))
    s_transmit_best = s_base * correction_phasor
    
    phi_learned = best_phi.detach().cpu().numpy()
    s_final_np = s_transmit_best.detach().cpu().numpy()


freqs = np.fft.fftshift(np.fft.fftfreq(N_fft, 1/Fs))


S_orig = np.fft.fftshift(np.fft.fft(s_base.numpy(), n=N_fft))
S_final = np.fft.fftshift(np.fft.fft(s_final_np, n=N_fft))


S_orig_db = 20*np.log10(np.abs(S_orig) + 1e-15)
S_final_db = 20*np.log10(np.abs(S_final) + 1e-15)


peak_ref = np.max(S_orig_db)
S_orig_norm = S_orig_db - peak_ref
S_final_norm = S_final_db - peak_ref


with torch.no_grad():
    # חישוב ב-PyTorch
    exact_vals = torch.matmul(steering_matrix, s_transmit_best)
    energies_torch = torch.abs(exact_vals)**2
    
    # המרה מסודרת ל-Numpy
    energies_np = energies_torch.detach().cpu().numpy()
    
    # חישוב dB ב-Numpy
    exact_db_absolute = 10 * np.log10(energies_np + 1e-40)
    
    # נרמול
    exact_db_relative = exact_db_absolute - peak_ref

print("--- True Null Depths (Calculated Mathematically) ---")
for k, f_val in enumerate(null_freqs_np):
    print(f"Freq {f_val/1e6} MHz: {exact_db_relative[k]:.2f} dB")


# 5. הציור
plt.figure(figsize=(12, 7))

# גרף האות המקורי
plt.plot(freqs/1e6, S_orig_norm, label='Original LFM', color='blue', alpha=0.3) # original signal

# גרף האות האופטימלי (FFT)
plt.plot(freqs/1e6, S_final_norm, label='Optimized Signal', color='red', linewidth=1) # optimized signal

# stars for exact null depths
plt.scatter(null_freqs_np/1e6, exact_db_relative, 
            color='black', marker='*', s=150, zorder=10, 
            label='Exact Analytical Depth')

# adding text next to stars
for k, f_val in enumerate(null_freqs_np):
    depth = exact_db_relative[k]
    plt.text(f_val/1e6, depth - 5, f"{depth:.1f} dB", 
             color='black', fontweight='bold', ha='center', fontsize=9)

plt.title("Normalized Power Spectrum [dB]")
plt.xlabel("Frequency [MHz]")

#plt.legend(loc='upper right')
plt.grid(True, alpha=0.5)
#plt.xlim(0, B/1e6 + 0.2)
#plt.ylim(-150, 5)
plt.tight_layout()
plt.show()

In [23]:
#4 - Matched Filter and PSLR Calculation
with torch.no_grad():
    correction_phasor = torch.complex(torch.cos(phi), torch.sin(phi))
    s_best = s_base * correction_phasor
    s_best_np = s_best.detach().numpy()
    s_base_np = s_base.numpy()
    s_base = torch.tensor(s_base).detach().clone()

mf_orig_db, pslr_orig, islr_orig = calculate_mf_pslr_islr_torch(s_best)
mf_new_db, pslr_new, islr_new = calculate_mf_pslr_islr_torch(s_base)
'''
plt.figure(figsize=(10, 6))
plt.plot(mf_orig_db, label=f'Original LFM (PSLR={pslr_orig:.1f}dB)', color='blue', alpha=0.6)
plt.plot(mf_new_db, label=f'Optimized LFM (PSLR={pslr_new:.1f}dB)', color='red', linestyle='--')

plt.title("Matched Filter Comparison")
plt.xlabel("index")
plt.ylabel("Amplitude [dB]")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
'''
print(f"--- PSLR Performence ---")
print(f"Selected Model based on Minimum Total Loss")
print(f"Original LFM PSLR:  {pslr_orig:.2f} dB")
print(f"Optimized LFM PSLR: {pslr_new:.2f} dB")
print(f"Degradation:        {pslr_orig - pslr_new:.2f} dB")
print("Notice, PSLR should be as high as possible, so positive degradation is bad.\n")


print(f"--- ISLR Performance ---")
print(f"Original LFM ISLR:  {islr_orig:.2f} dB")
print(f"Optimized LFM ISLR: {islr_new:.2f} dB")
print(f"Degradation:        {islr_orig - islr_new:.2f} dB")
print("Notice, ISLR should be as small as possible, so negative degradation is bad.\n")

# lags = np.arange(-len(s_base_np) + 1, len(s_base_np))
# time_lags = (lags / Fs) * 1e6 



--- PSLR Performence ---
Selected Model based on Minimum Total Loss
Original LFM PSLR:  13.43 dB
Optimized LFM PSLR: 13.46 dB
Degradation:        -0.03 dB
Notice, PSLR should be as high as possible, so positive degradation is bad.

--- ISLR Performance ---
Original LFM ISLR:  -9.56 dB
Optimized LFM ISLR: -9.83 dB
Degradation:        0.26 dB
Notice, ISLR should be as small as possible, so negative degradation is bad.



C:\Users\User\AppData\Local\Temp\ipykernel_11800\2886784997.py:7: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [ ]:
#5 Saving phi
phi_numpy = best_phi.detach().cpu().numpy()


correction_phasor = np.exp(1j * phi_numpy)


filename = 'optimal_phasor.npy'
np.save(filename, correction_phasor)

print(f"Success! Saved optimized phasor to '{filename}'")
print(f"File shape: {correction_phasor.shape}")

Success! Saved optimized phasor to 'optimal_phasor.npy'
File shape: (600,)


In [25]:
#The first One 
import torch
import numpy as np
import optuna

def objective(trial):
    
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    # b_smooth = trial.suggest_float("beta_smooth", 1e-5, 1.0, log=True)
    b_norm = trial.suggest_float("beta_norm", 1e-5, 1.0, log=True)
    # beta_div1 = trial.suggest_float("beta_div1", 1e-5, 1.0, log=True)
    # beta_div2 = trial.suggest_float("beta_div2", 1e-5, 1.0, log=True)
    
    phi = torch.zeros(N, requires_grad=True, dtype=torch.float64)
    optimizer = torch.optim.Adam([phi], lr=lr)
    
    best_loss = float('inf')
    iterations = 5000 

    
    for i in range(iterations):
        correction_phasor = torch.complex(torch.cos(phi), torch.sin(phi))
        s_transmit = s_base * correction_phasor
        
        spectral_vals = torch.matmul(steering_matrix, s_transmit)
        energies = torch.abs(spectral_vals)**2
        
        loss_nulls = torch.sum(energies) 
        #diff_phi = phi[1:] - phi[:-1]
        #reg_term = b_smooth * torch.sum(diff_phi**2)
        loss_norm = b_norm * torch.sum(phi**2) 
        if((beta_div1) and (beta_div2)):  
            integrals1 = [
                torch.trapezoid(z[:, k] * correction_phasor * t_norm, x=t_norm) 
                for k in range(z.shape[1])
            ]
            loss_div1 = beta_div1 * sum([torch.abs(val)**2 for val in integrals1])    
            integrals2 = [
                torch.trapezoid(z[:, k] * correction_phasor * (t_norm**2), x=t_norm) 
                for k in range(z.shape[1])
            ]
            loss_div2 = beta_div2 * sum([torch.abs(val)**2 for val in integrals2])
        else:
            loss_div1 = 0
            loss_div2 = 0
            
        loss = loss_nulls  + loss_norm + loss_div1 + loss_div2 # + reg_term
        
        
        if loss.item() < best_loss:
            best_loss = loss.item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return best_loss


#study = optuna.create_study(direction="minimize")
#study.optimize(objective, n_trials=30) 

#print("Best Parameters found:", study.best_params)

In [ ]:
import optuna
import torch

def objective(trial):
    
    #lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    beta_smooth = trial.suggest_float("beta_smooth", 1e-6, 1e-3, log=True)
    beta_norm = trial.suggest_float("beta_norm", 1e-6, 1e-3, log=True)
    beta_pslr = trial.suggest_float("beta_pslr", 1e-4, 1.0, log=True)
    beta_islr = trial.suggest_float("beta_islr", 1e-4, 1.0, log=True)
    beta_div1 = trial.suggest_float("beta_div1", 1e-6, 1e-1, log=True)
    beta_div2 = trial.suggest_float("beta_div2", 1e-6, 1e-1, log=True) 

    phi = torch.tensor(phi_start, dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.Adam([phi], lr=0.01)
    
    
    for i in range(2500): 
        correction_phasor = torch.complex(torch.cos(phi), torch.sin(phi))
        s_transmit = s_base * correction_phasor
        
        spectral_vals = torch.matmul(steering_matrix, s_transmit)
        energies = torch.abs(spectral_vals)**2
        
       
        loss_nulls = torch.sum(energies) 
        
        diff_phi = phi[1:] - phi[:-1]
        reg_term = beta_smooth * torch.sum(diff_phi**2)
        loss_norm = beta_norm * torch.sum(phi**2)
        
        
        if beta_div1 > 0 or beta_div2 > 0:
            integrals1 = [
                torch.trapezoid(z[:, k] * correction_phasor * t_norm, x=t_norm) 
                for k in range(z.shape[1])
            ]
            loss_div1 = beta_div1 * sum([torch.abs(val)**2 for val in integrals1])

            integrals2 = [
                torch.trapezoid(z[:, k] * correction_phasor * (t_norm**2), x=t_norm) 
                for k in range(z.shape[1])
            ]
            loss_div2 = beta_div2 * sum([torch.abs(val)**2 for val in integrals2])
        else:
            loss_div1 = 0
            loss_div2 = 0
        
        
        mf_db, pslr, islr = calculate_mf_pslr_islr_torch(s_transmit)
        loss_pslr = -1 * beta_pslr * pslr
        loss_islr = beta_islr * islr 

        
        loss = loss_norm + loss_div1 + loss_div2 + loss_nulls + loss_pslr + loss_islr + reg_term
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    
    
    final_null_energy = loss_nulls.item()
    final_pslr_val = pslr.item()
    final_islr_val = islr.item()

    return final_null_energy, final_pslr_val, final_islr_val


study = optuna.create_study(directions=["minimize", "maximize", "minimize"])
study.optimize(objective, n_trials=50)



[I 2026-02-25 15:25:09,883] A new study created in memory with name: no-name-41b4a949-2865-4d38-a527-d7ba21068974
C:\Users\User\AppData\Local\Temp\ipykernel_11800\866495995.py:14: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).

[I 2026-02-25 15:25:17,659] Trial 0 finished with values: [0.030940988825294502, 13.356407199286828, -9.379839209197112] and parameters: {'beta_smooth': 2.719030749506329e-06, 'beta_norm': 2.6867658607236302e-05, 'beta_pslr': 0.0009434257377185918, 'beta_islr': 0.0003346792117285141, 'beta_div1': 1.5808288115134017e-06, 'beta_div2': 9.940375335511337e-05}.
[I 2026-02-25 15:25:25,673] Trial 1 finished with values: [0.00017742981769083884, 14.15994896308753, -9.956388596547882] and parameters: {'beta_smooth': 1.2944943990899182e-05, 'beta_norm': 1.8791475495802712e-05, 'beta_pslr': 0.03395104541880879, 'beta_islr': 0

In [ ]:
import optuna.visualization as vis

fig = vis.plot_pareto_front(
    study, 
    target_names=["Null Energy (Min)", "PSLR (Max)", "ISLR (Min)"]
)
fig.show()


print("\n--- Best Options on the Pareto Front ---")
best_trials = study.best_trials

for i, trial in enumerate(best_trials):
    print(f"\nOption {i+1} (Trial #{trial.number}):")
    print(f"  Null Energy: {trial.values[0]:.6f}")
    print(f"  PSLR:        {trial.values[1]:.6f} dB")
    print(f"  ISLR:        {trial.values[2]:.6f} dB")
    print("  Betas to use:")
    for key, value in trial.params.items():
        print(f"    {key} = {value:.6e}")


--- Best Options on the Pareto Front ---

Option 1 (Trial #3):
  Null Energy: 0.014111
  PSLR:        41.789908 dB
  ISLR:        -20.013476 dB
  Betas to use:
    beta_smooth = 3.441934e-06
    beta_norm = 1.552258e-04
    beta_pslr = 4.268689e-01
    beta_islr = 9.919571e-01
    beta_div1 = 3.255936e-02
    beta_div2 = 8.529307e-02

Option 2 (Trial #5):
  Null Energy: 0.004994
  PSLR:        27.808739 dB
  ISLR:        -21.367374 dB
  Betas to use:
    beta_smooth = 1.169594e-06
    beta_norm = 7.343675e-06
    beta_pslr = 4.734573e-04
    beta_islr = 9.881440e-01
    beta_div1 = 7.197988e-02
    beta_div2 = 2.532424e-06

Option 3 (Trial #6):
  Null Energy: 0.000000
  PSLR:        16.217262 dB
  ISLR:        -10.495679 dB
  Betas to use:
    beta_smooth = 9.077519e-05
    beta_norm = 8.432383e-04
    beta_pslr = 7.597650e-02
    beta_islr = 3.117110e-03
    beta_div1 = 7.517796e-05
    beta_div2 = 2.938032e-06

Option 4 (Trial #7):
  Null Energy: 0.000082
  PSLR:        29.683428 dB

In [ ]:
pip install plotly
pip install --upgrade nbformat